# Modellvergleich\n\nTrainiert alle vier bisherigen Modelle (`poisson_baseline`, `form_poisson`, `shots_form_poisson`, `odds_poisson`) auf 21/22-24/25 und vergleicht sie Spieltag für Spieltag auf Saison 25/26, mit denselben Kicktipp-Regeln (4 = exakt, 3 = richtige Tordifferenz, 2 = nur Tendenz, 0 = daneben). Zur Einordnung: manuelles Tippen kam auf 400 Punkte, der Sieger der Runde auf 450.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.csv_source import CSVSource
from src.data.loader import DataLoader
from src.models.poisson_regressor import PoissonRegressor
from src.models.scoring import kicktipp_points

from src.models.poisson_baseline.features import build_design_matrix as baseline_build_X
from src.models.poisson_baseline.predict import predict_score as baseline_predict

from src.models.form_poisson.features import build_design_matrix as form_build_X, compute_current_form as form_current_form
from src.models.form_poisson.predict import predict_score as form_predict

from src.models.shots_form_poisson.features import build_design_matrix as shots_build_X, compute_current_form as shots_current_form
from src.models.shots_form_poisson.predict import predict_score as shots_predict

from src.models.odds_poisson.features import build_design_matrix as odds_build_X
from src.models.odds_poisson.predict import predict_score as odds_predict

HUMAN_POINTS = 400
WINNER_POINTS = 450

## Daten laden\n\nZwei Quellen: `data/raw/` (datahub.io, für baseline/form/shots-form) und `data/raw/odds/` (football-data.co.uk, gleiche Spiele plus Quoten-Spalten, aber anderes Datumsformat - daher separat konvertiert).

In [ ]:
TRAIN_SEASONS = ["season-2122.csv", "season-2223.csv", "season-2324.csv", "season-2425.csv"]

DATA_DIR = Path.cwd().parent / "data" / "raw"
train_df = pd.concat(
    [DataLoader(CSVSource(DATA_DIR / f)).load() for f in TRAIN_SEASONS], ignore_index=True
)
test_df = DataLoader(CSVSource(DATA_DIR / "season-2526.csv")).load()
test_df = test_df.sort_values("date", kind="stable").reset_index(drop=True)
test_df["matchday"] = test_df.index // 9 + 1

DATA_DIR_ODDS = Path.cwd().parent / "data" / "raw" / "odds"
train_odds = pd.concat(
    [DataLoader(CSVSource(DATA_DIR_ODDS / f)).load() for f in TRAIN_SEASONS], ignore_index=True
)
test_odds = DataLoader(CSVSource(DATA_DIR_ODDS / "season-2526.csv")).load()
test_odds["date"] = pd.to_datetime(test_odds["date"], format="%d/%m/%Y")
test_odds = test_odds.sort_values("date", kind="stable").reset_index(drop=True)
test_odds["matchday"] = test_odds.index // 9 + 1

train_df.shape, test_df.shape, train_odds.shape, test_odds.shape

## Baseline

In [ ]:
X, y, team_index = baseline_build_X(train_df)
baseline_model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
baseline_model.fit(X, y)

baseline_results = []
for match in test_df.itertuples():
    predicted = baseline_predict(baseline_model, match.hometeam, match.awayteam, team_index)
    actual = (int(match.fthg), int(match.ftag))
    baseline_results.append({
        "model": "baseline",
        "matchday": match.matchday,
        "points": kicktipp_points(predicted, actual),
    })

print("baseline loss:", baseline_model.loss_history_[-1])

## Tore-Formkurve (`form_poisson`)

In [ ]:
FORM_WINDOW_GOALS = 10

X, y, form_team_index = form_build_X(train_df, form_window=FORM_WINDOW_GOALS)
form_model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
form_model.fit(X, y)

form_results = []
history_df = train_df.copy()
for matchday in range(1, test_df["matchday"].max() + 1):
    matchday_matches = test_df[test_df["matchday"] == matchday]
    current_form = form_current_form(history_df, form_window=FORM_WINDOW_GOALS)
    for match in matchday_matches.itertuples():
        predicted = form_predict(form_model, match.hometeam, match.awayteam, form_team_index, current_form)
        actual = (int(match.fthg), int(match.ftag))
        form_results.append({"model": "form (goals)", "matchday": matchday, "points": kicktipp_points(predicted, actual)})
    history_df = pd.concat([history_df, matchday_matches], ignore_index=True)

print("form (goals) loss:", form_model.loss_history_[-1])

## Schuss-Formkurve (`shots_form_poisson`)

In [ ]:
FORM_WINDOW_SHOTS = 5

X, y, shots_team_index = shots_build_X(train_df, form_window=FORM_WINDOW_SHOTS)
shots_model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
shots_model.fit(X, y)

shots_results = []
history_df = train_df.copy()
for matchday in range(1, test_df["matchday"].max() + 1):
    matchday_matches = test_df[test_df["matchday"] == matchday]
    current_form = shots_current_form(history_df, form_window=FORM_WINDOW_SHOTS)
    for match in matchday_matches.itertuples():
        predicted = shots_predict(shots_model, match.hometeam, match.awayteam, shots_team_index, current_form)
        actual = (int(match.fthg), int(match.ftag))
        shots_results.append({"model": "shots-form", "matchday": matchday, "points": kicktipp_points(predicted, actual)})
    history_df = pd.concat([history_df, matchday_matches], ignore_index=True)

print("shots-form loss:", shots_model.loss_history_[-1])

## Quoten (`odds_poisson`)

In [ ]:
X, y, _ = odds_build_X(train_odds)
odds_model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
odds_model.fit(X, y)

odds_results = []
for match in test_odds.itertuples():
    predicted = odds_predict(odds_model, match.avgh, match.avgd, match.avga)
    actual = (int(match.fthg), int(match.ftag))
    odds_results.append({"model": "odds", "matchday": match.matchday, "points": kicktipp_points(predicted, actual)})

print("odds loss:", odds_model.loss_history_[-1])

## Zusammenfassung

In [ ]:
all_results = pd.concat([
    pd.DataFrame(baseline_results),
    pd.DataFrame(form_results),
    pd.DataFrame(shots_results),
    pd.DataFrame(odds_results),
], ignore_index=True)

summary = (
    all_results
    .groupby("model")["points"]
    .agg(gesamt="sum", pro_spiel="mean")
    .sort_values("gesamt", ascending=False)
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = summary["gesamt"].plot.bar(ax=ax, color="steelblue")
ax.axhline(HUMAN_POINTS, color="orange", linestyle="--", label=f"Du (manuell): {HUMAN_POINTS}")
ax.axhline(WINNER_POINTS, color="red", linestyle="--", label=f"Sieger der Runde: {WINNER_POINTS}")
ax.set_ylabel("Gesamtpunkte (Saison 25/26)")
ax.set_title("Modellvergleich: Gesamtpunkte")
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for model_name, group in all_results.groupby("model"):
    cumulative = group.groupby("matchday")["points"].sum().cumsum()
    ax.plot(cumulative.index, cumulative.values, marker="o", markersize=3, label=model_name)

ax.axhline(HUMAN_POINTS, color="orange", linestyle="--", label=f"Du (manuell): {HUMAN_POINTS}")
ax.axhline(WINNER_POINTS, color="red", linestyle="--", label=f"Sieger der Runde: {WINNER_POINTS}")
ax.set_xlabel("Spieltag")
ax.set_ylabel("Kumulierte Punkte")
ax.set_title("Punkteverlauf über die Saison 25/26")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def categorize(points: int) -> str:
    if points == 4:
        return "exakt"
    if points in (2, 3):
        return "tendenz"
    return "falsch"

all_results["category"] = all_results["points"].apply(categorize)

category_summary = (
    all_results
    .groupby(["model", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["exakt", "tendenz", "falsch"], fill_value=0)
    .loc[summary.index]
)
category_summary